# 02 Silver enrich

Adds clean columns to silver for the front end toggles, and builds a cleaned tags table.

**This notebook:**
1. Checks what each category value really means
2. Splits category into two columns: gender and line
3. Explodes tags into their own table, drops generic tags and merges synonyms
4. Validates everything

### Check what category means
Compares category against the word "menswear" in the filename, to confirm whether Ready-to-Wear is womenswear and Menswear is men's ready-to-wear. Also checks what the null categories are.

In [0]:
from pyspark.sql import functions as F

s = spark.table("workspace.silhouette_archive.silver_runway")
s = s.withColumn("file_says_menswear", F.lower("filename").contains("menswear"))

display(s.groupBy("category", "file_says_menswear").count().orderBy("category"))

display(s.filter(F.col("category").isNull())
         .select("designer", "season", "year", "filename")
         .limit(20))

In [0]:
display(s.filter(F.col("category").isNull())
         .groupBy("season", "file_says_menswear")
         .count()
         .orderBy("count", ascending=False))

### Add gender and line
- **gender:** men if category is Menswear or the filename says menswear, otherwise women. Checks showed every null category is womenswear.
- **line:** couture, bridal, or ready-to-wear. Null categories are Resort and Pre-Fall collections, which are ready-to-wear.

In [0]:
enriched = (s
    .withColumn("gender",
        F.when((F.col("category") == "Menswear") | F.col("file_says_menswear"), "men")
         .otherwise("women"))
    .withColumn("line",
        F.when(F.col("category") == "Couture", "couture")
         .when(F.col("category") == "Bridal", "bridal")
         .otherwise("ready-to-wear"))
    .drop("file_says_menswear"))

enriched.write.mode("overwrite").saveAsTable("workspace.silhouette_archive.silver_runway_enriched")
display(spark.table("workspace.silhouette_archive.silver_runway_enriched")
        .groupBy("gender", "line").count().orderBy("gender", "line"))

### Validate gender and line
Checks the new columns only contain expected values, and that no rows were lost.

In [0]:
e = spark.table("workspace.silhouette_archive.silver_runway_enriched")

genders = {r.gender for r in e.select("gender").distinct().collect()}
lines = {r.line for r in e.select("line").distinct().collect()}
assert genders <= {"men", "women"}, f"Unexpected genders: {genders}"
assert lines <= {"ready-to-wear", "couture", "bridal"}, f"Unexpected lines: {lines}"
assert e.count() == s.count(), "Row count changed"

print(f"All checks passed: {e.count():,} rows")

### Explore all tags
Explodes the tags list into one row per image per tag, then counts every distinct tag. Used to decide which tags are too generic to keep and which are synonyms.

In [0]:
e = spark.table("workspace.silhouette_archive.silver_runway_enriched")

tags = e.select("row_id", F.explode("tags").alias("tag"))
tag_counts = tags.groupBy("tag").count().orderBy("count", ascending=False)

print("Distinct tags:", tag_counts.count())
display(tag_counts)

In [0]:
print(", ".join(f"{r.tag} ({r['count']})" for r in tag_counts.limit(600).collect()))

In [0]:
rows = tag_counts.limit(1000).collect()[600:]
print(", ".join(f"{r.tag} ({r['count']})" for r in rows))

### Clean tags into silver_tags

The raw `tags` column comes from automatic image labelling, so it mixes garments with noise: generic labels (Human, Clothing), model names, and random objects (Bird, Handrail). There are about 11,000 distinct raw tags, and most appear only a handful of times.

**Approach: a keep list, not a drop list.**
1. Reviewed the top 1,000 raw tags by count and picked only fashion ones: garments, footwear, bags, headwear, accessories, materials, patterns and themes.
2. Merged synonyms into one clean name, for example Overcoat → Coat, Footwear and Shoe → Shoes, Tartan → Plaid, Ballerina → Ballet.
3. Anything not on the list is dropped automatically.

**Tables written:**
- `tag_map`: the rule itself, raw_tag → tag. Stored as a table so it's visible and easy to edit.
- `silver_tags`: one row per image per clean tag, with duplicates removed (an image tagged both Shoe and Footwear counts once as Shoes).

**Result:** 101 clean tags, each on at least 51 images. These become the clickable chips in the front end.

In [0]:
from pyspark.sql import functions as F

TAG_MAP = {
    # Outerwear
    "Coat": "Coat", "Overcoat": "Coat", "Trench Coat": "Trench Coat", "Raincoat": "Raincoat",
    "Jacket": "Jacket", "Leather Jacket": "Leather Jacket", "Blazer": "Blazer",
    "Cloak": "Cape", "Cape": "Cape", "Poncho": "Poncho",
    "Sweatshirt": "Sweatshirt", "Hoodie": "Sweatshirt", "Fleece": "Fleece",
    # Dresses and robes
    "Dress": "Dress", "Evening Dress": "Dress", "Gown": "Gown", "Robe": "Gown",
    "Wedding Gown": "Wedding Dress", "Kimono": "Kimono", "Sari": "Sari",
    # Tops
    "Shirt": "Shirt", "Dress Shirt": "Shirt", "Blouse": "Blouse", "TShirt": "T-Shirt",
    "Tank Top": "Tank Top", "Undershirt": "Tank Top", "Jersey": "Jersey",
    "Sweater": "Knitwear", "Cardigan": "Knitwear", "Knitting": "Knitwear",
    "Vest": "Vest", "Corset": "Corset",
    "Lingerie": "Lingerie", "Bra": "Lingerie", "Underwear": "Lingerie", "Panties": "Lingerie",
    "Swimwear": "Swimwear", "Bikini": "Swimwear",
    # Bottoms
    "Pants": "Trousers", "Jeans": "Jeans", "Shorts": "Shorts",
    "Skirt": "Skirt", "Miniskirt": "Miniskirt", "Kilt": "Kilt",
    "Tights": "Tights", "Pantyhose": "Tights", "Sock": "Socks",
    # Tailoring and uniforms
    "Suit": "Suit", "Tuxedo": "Suit", "Tie": "Tie", "Necktie": "Tie", "Suspenders": "Suspenders",
    "Military Uniform": "Military", "Military": "Military", "Army": "Military",
    "Sailor Suit": "Sailor", "Pajamas": "Pyjamas", "Apron": "Apron",
    "Armor": "Armour", "Chain Mail": "Armour", "Armored": "Armour",
    "Latex Clothing": "Latex", "Spandex": "Spandex",
    # Footwear
    "Footwear": "Shoes", "Shoe": "Shoes", "Boot": "Boots", "Riding Boot": "Riding Boots",
    "Cowboy Boot": "Cowboy Boots", "High Heel": "Heels", "Heel": "Heels",
    "Sandal": "Sandals", "FlipFlop": "Sandals", "Sneaker": "Sneakers", "Running Shoe": "Sneakers",
    "Clogs": "Clogs", "Wedge": "Wedges",
    # Bags
    "Bag": "Bag", "Handbag": "Bag", "Purse": "Bag", "Tote Bag": "Tote",
    "Backpack": "Backpack", "Briefcase": "Briefcase", "Wallet": "Wallet",
    # Headwear
    "Hat": "Hat", "Sun Hat": "Sun Hat", "Cap": "Cap", "Baseball Cap": "Cap",
    "Beanie": "Beanie", "Cowboy Hat": "Cowboy Hat", "Bonnet": "Bonnet", "Turban": "Turban",
    "Helmet": "Helmet", "Crash Helmet": "Helmet", "Hardhat": "Helmet",
    "Headband": "Headband", "Veil": "Veil", "Hood": "Hood", "Bandana": "Bandana", "Tiara": "Tiara",
    # Other accessories
    "Sunglasses": "Sunglasses", "Glasses": "Glasses", "Goggles": "Goggles",
    "Scarf": "Scarf", "Stole": "Scarf", "Feather Boa": "Feather Boa",
    "Belt": "Belt", "Buckle": "Belt", "Sash": "Sash", "Glove": "Gloves", "Bow": "Bow",
    "Zipper": "Zipper", "Wristwatch": "Watch", "Pearl": "Pearls",
    "Jewelry": "Jewellery", "Necklace": "Jewellery", "Pendant": "Jewellery",
    "Earring": "Jewellery", "Ring": "Jewellery", "Diamond": "Jewellery",
    "Gemstone": "Jewellery", "Crystal": "Jewellery", "Bead": "Jewellery", "Anklet": "Jewellery",
    "Bangles": "Jewellery", "Bead Necklace": "Jewellery",
    # Materials
    "Fur": "Fur", "Lace": "Lace", "Linen": "Linen", "Silk": "Silk",
    "Velvet": "Velvet", "Denim": "Denim", "Suede": "Suede",
    # Patterns and colour
    "Plaid": "Plaid", "Tartan": "Plaid", "Polka Dot": "Polka Dot", "Leopard": "Leopard",
    "Paisley": "Paisley", "Camouflage": "Camouflage", "Floral Design": "Floral",
    "Embroidery": "Embroidery", "Khaki": "Khaki",
    # Themes
    "Flamenco": "Flamenco", "Hippie": "Hippie", "Ballet": "Ballet", "Ballerina": "Ballet",
}

print(len(TAG_MAP), "raw tags mapped to", len(set(TAG_MAP.values())), "clean tags")

e = spark.table("workspace.silhouette_archive.silver_runway_enriched")
tags = e.select("row_id", F.explode("tags").alias("tag"))

mapping = spark.createDataFrame(list(TAG_MAP.items()), ["raw_tag", "tag"])
mapping.write.mode("overwrite").saveAsTable("workspace.silhouette_archive.tag_map")

silver_tags = (tags.withColumnRenamed("tag", "raw_tag")
    .join(mapping, "raw_tag")
    .select("row_id", "tag")
    .dropDuplicates(["row_id", "tag"]))

silver_tags.write.mode("overwrite").saveAsTable("workspace.silhouette_archive.silver_tags")
display(spark.table("workspace.silhouette_archive.silver_tags")
        .groupBy("tag").count().orderBy("count", ascending=False))

### Validate silver_tags
Stops with an error if any check fails:
- Every row_id exists in silver_runway_enriched
- No nulls in row_id or tag
- No duplicate image and tag pairs
- Every tag is a clean name from tag_map
- No generic tags left

In [0]:
st = spark.table("workspace.silhouette_archive.silver_tags")
e = spark.table("workspace.silhouette_archive.silver_runway_enriched")
tm = spark.table("workspace.silhouette_archive.tag_map")

orphans = st.join(e.select("row_id"), "row_id", "left_anti").count()
assert orphans == 0, f"{orphans} tag rows point to missing images"

assert st.filter(F.col("row_id").isNull() | F.col("tag").isNull()).count() == 0, "Nulls found"

n = st.count()
assert st.select("row_id", "tag").distinct().count() == n, "Duplicate image and tag pairs"

clean = {r.tag for r in tm.select("tag").distinct().collect()}
found = {r.tag for r in st.select("tag").distinct().collect()}
assert found <= clean, f"Unexpected tags: {found - clean}"

generic = {"Human", "Person", "Clothing", "Apparel", "Fashion", "Runway"}
assert not (found & generic), f"Generic tags left: {found & generic}"

print(f"All checks passed: {n:,} tag rows across {len(found)} tags")